# HydraY NNUE - bucket 0, secondo giro: 1,18B + 103M di finali

Runtime → Cambia tipo di runtime → **GPU (T4)**, poi Runtime → **Esegui tutte**.
Durata ~1h. **Non lasciare la scheda inattiva.**

### A cosa serve
Il primo batch di finali (50,7M) ha portato il bucket 0 dallo 0,23% al 3,7% e ha
dato **+27,94 ±11,26 Elo**, con KQvK da 128 a 522 cp. Restava pero' sotto: il
valore vero di donna contro re nudo e' dell'ordine dei 900, e il bucket 0 era
ancora sotto-rappresentato di un fattore 3 rispetto al 12,5% uniforme.

Questo raddoppia il batch: **103.450.190 posizioni di soli finali** (86% bucket
0), che portano il bucket 0 al **7,1%**. Verificato uniforme su entrambe le meta'
del file, quindi nessun accumulo in coda.

Il secondo batch e' etichettato dalla rete +27,9 appena spedita, il primo dalla
3.0.0: etichette leggermente migliori sulla meta' nuova.

### Il controllo che conta, prima dello SPRT
Alla fine il notebook stampa i sanity eval. **KQvK deve salire sopra i 522 cp**
della rete attuale. Se resta fermo li', il secondo batch non ha aggiunto niente e
lo sai in un minuto invece che in quattro ore di partite.

### Perche' due parti
Il file scompattato e' 38,3 GiB e il mount Drive tiene in cache cio' che legge
finche' il processo non esce: un unico .zst darebbe un picco di 61 GiB contro i
~65 disponibili, un margine troppo sottile. Due parti scompattate in sequenza
tengono il picco a **~50 GiB**.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    # L'output va riletto e ristampato da Python: subprocess.run() senza capture
    # scrive sui file descriptor del KERNEL, che Colab non mostra nella cella.
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + dataset ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]
P1, P2 = find('hydray_v6_part1.bin.zst'), find('hydray_v6_part2.bin.zst')
os.environ['P1'], os.environ['P2'] = P1, P2
print('parte 1:', P1, os.path.getsize(P1), 'byte')
print('parte 2:', P2, os.path.getsize(P2), 'byte')

NET_ID   = 'hydray-eg103m'
TOTAL_SB = 40
TRAINER  = '/content/th/nnue/trainer'

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto.
sh('rm -rf /content/th')
sh('git clone --depth 1 --branch dev https://github.com/ThomasGhione/HydraY /content/th')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const HIDDEN: usize = 512;' in src, 'atteso lo strato nascosto a 512 (la 1024 ha perso)'
assert 'const INPUT_BUCKETS: usize = 4;' in src, 'attesi 4 king bucket'
print('branch dev, 512 neuroni, 4 king bucket: ok')

In [ ]:
# --- decompressione in due parti (ogni zstd esce e libera la cache di Drive) ---
sh('apt-get -qq install -y zstd >/dev/null')

TOT, HALF = 41070409664, 20535204832
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'liberi {free_gb:.1f} GiB, picco atteso ~50 GiB')
assert free_gb > 52, 'disco insufficiente'

sh('zstd -d -T0 --long=27 -c "$P1" > /content/data.bin')
assert os.path.getsize('/content/data.bin') == HALF, 'parte 1 di taglia inattesa'
print('parte 1 ok'); sh('df -h /content | tail -1')

sh('zstd -d -T0 --long=27 -c "$P2" >> /content/data.bin')
SIZE = os.path.getsize('/content/data.bin')
print('data.bin:', SIZE, 'byte =', SIZE // 32, 'posizioni')
assert SIZE == TOT, f'taglia finale inattesa: {SIZE}'
sh('df -h /content | tail -1')

In [ ]:
# --- validation set: 1000 MiB dalla coda, mai addestrati ---
TEST_MIB = 1000
skip_mib = SIZE // (1024*1024) - TEST_MIB
sh(f'dd if=/content/data.bin bs=1M skip={skip_mib} count={TEST_MIB} of=/content/test.bin status=progress')
print('test.bin:', os.path.getsize('/content/test.bin'), 'byte')

In [ ]:
# --- training: un solo run, 40 superbatch ---
sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda '
   f'TEST_PATH=/content/test.bin '
   f'cargo run -r --bin trainer --features cuda -- '
   f'/content/data.bin {TOTAL_SB} {NET_ID}')

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
assert sz == 3163200, f'taglia {sz}: attesa 3.163.200 (512 neuroni, 4 king bucket)'
print('quantised.bin:', sz, 'byte - architettura confermata\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*60)
print('GUARDA LA RIGA "KQvK" QUI SOPRA.')
print('  La rete ATTUALE (batch da 50M) da 522 cp. Il valore vero e ~900.')
print('  Se e fermo a ~522, raddoppiare il batch non ha aggiunto niente.')
print('='*60)

## Come leggere il risultato

**Primo controllo, immediato: KQvK.** La 3.0.0 dava 128 cp. Se il batch di
finali ha funzionato quel numero deve salire in modo netto. Se resta sotto i 200,
l'esperimento e' fallito e non vale la pena spendere lo SPRT.

**Secondo: gli altri sanity eval non devono peggiorare.** startpos ~+50,
mediogioco ~900, donna in piu' ~1700. Il feature transformer `l0` e' condiviso
fra tutti i bucket, quindi il 4% di posizioni di finale in piu' potrebbe in
teoria spostare la rappresentazione a scapito del mediogioco. Se quei numeri
crollano, il prezzo e' troppo alto.

**Terzo: la validation loss.** Non e' confrontabile con quella dei run
precedenti - il test set e' diverso, perche' contiene anche finali.

Poi tocca allo SPRT testa a testa contro la rete della 3.0.0.
